# NLP Preprocessing Pipeline
**End-to-end pipeline:** Kaggle download → EDA → Text Cleaning → BERT Tokenization → Stratified Split → Export

---

## 0. Setup & Dependencies

In [ ]:
# Install required packages (run once)
# !pip install kaggle transformers datasets scikit-learn pandas matplotlib seaborn wordcloud

In [ ]:
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import kagglehub
import pandas as pd

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('✅ All libraries imported successfully')

## 1. Configuration

In [ ]:
# ─── USER CONFIG ──────────────────────────────────────────────────────────────
KAGGLE_DATASET   = 'nn_final/sentiment140'   # <-- change to your dataset slug
TEXT_COL         = 'statement'                    # column with raw text
LABEL_COL        = 'status'               # column with class labels
BERT_MODEL       = 'bert-base-uncased'       # any HuggingFace tokenizer
MAX_LENGTH       = 128                       # token truncation length

TRAIN_RATIO      = 0.70
VAL_RATIO        = 0.15
TEST_RATIO       = 0.15
RANDOM_STATE     = 42

DATA_DIR         = Path('data')
OUTPUT_DIR       = Path('output')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, 'Ratios must sum to 1.0'
print(f'Config loaded  |  dataset: {KAGGLE_DATASET}  |  model: {BERT_MODEL}  |  max_len: {MAX_LENGTH}')

## 2. Download Dataset from Kaggle

> **Authentication:** Place your `kaggle.json` API key in `~/.kaggle/kaggle.json`  
> Get it from **kaggle.com → Account → Create New API Token**

In [ ]:
# Download latest version
path = kagglehub.dataset_download("suchintikasarkar/sentiment-analysis-for-mental-health")

files = os.listdir(path=path)

df = pd.read_csv(os.path.join(path, files[0]))

In [ ]:
print(f'Shape : {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# ── Normalise to (text, label) if needed ──────────────────────────────────────
# Map numeric labels → readable strings  (edit to match your dataset)
LABEL_MAP = {0: 'negative', 2: 'neutral', 4: 'positive'}

df = df[[TEXT_COL, LABEL_COL]].copy()
if df[LABEL_COL].dtype != object:
    df[LABEL_COL] = df[LABEL_COL].map(LABEL_MAP).fillna(df[LABEL_COL].astype(str))

print(f'Working dataframe: {df.shape}')
print(f'Label values     : {sorted(df[LABEL_COL].unique())}')
df.head(3)

## 3. Exploratory Data Analysis

### 3.1 Class Distribution

In [ ]:
label_counts = df[LABEL_COL].value_counts().sort_index()
label_pct    = label_counts / len(df) * 100

print('Class distribution')
print('─' * 38)
for cls, cnt in label_counts.items():
    bar = '█' * int(label_pct[cls] / 2)
    print(f'{str(cls):>12}  {cnt:>8,}  ({label_pct[cls]:5.1f}%)  {bar}')
print('─' * 38)
print(f'{"TOTAL":>12}  {len(df):>8,}  (100.0%)')

# Imbalance ratio
imbalance = label_counts.max() / label_counts.min()
print(f'\nImbalance ratio (max/min): {imbalance:.2f}x')
if imbalance > 2:
    print('⚠  Consider class-weighting or oversampling during training.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(label_counts.index.astype(str), label_counts.values,
            color=sns.color_palette('muted', len(label_counts)), edgecolor='white')
axes[0].set_title('Class Counts', fontweight='bold')
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + len(df)*0.003,
                 f'{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)

# Pie chart
axes[1].pie(label_counts.values, labels=label_counts.index.astype(str),
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('muted', len(label_counts)))
axes[1].set_title('Class Proportions', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', bbox_inches='tight')
plt.show()
print('Figure saved → output/class_distribution.png')

### 3.2 Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %':     missing_pct.round(4)
})
print('Missing value audit')
print(missing_df.to_string())

empty_text = (df[TEXT_COL].astype(str).str.strip() == '').sum()
print(f'\nEmpty / whitespace-only text rows: {empty_text:,}')

In [ ]:
# Drop rows with missing text or label
before = len(df)
df.dropna(subset=[TEXT_COL, LABEL_COL], inplace=True)
df = df[df[TEXT_COL].astype(str).str.strip() != ''].copy()
df.reset_index(drop=True, inplace=True)
print(f'Rows removed (missing): {before - len(df):,}  |  Remaining: {len(df):,}')

### 3.3 Duplicate Detection

In [ ]:
n_exact_dup    = df.duplicated(subset=[TEXT_COL]).sum()
n_full_dup     = df.duplicated().sum()
n_cross_label  = df[df.duplicated(subset=[TEXT_COL], keep=False)] \
                   .groupby(TEXT_COL)[LABEL_COL].nunique()
n_conflict     = (n_cross_label > 1).sum()

print(f'Duplicate text rows (same text, any label) : {n_exact_dup:,}')
print(f'Fully duplicate rows (text + label)        : {n_full_dup:,}')
print(f'Conflicting label rows (same text, diff lbl): {n_conflict:,}')

if n_conflict > 0:
    print('\n⚠  Conflicting labels — showing top 5 examples:')
    conflict_texts = n_cross_label[n_cross_label > 1].index[:5]
    print(df[df[TEXT_COL].isin(conflict_texts)][[TEXT_COL, LABEL_COL]].to_string())

In [ ]:
# Remove exact duplicates (keep first occurrence)
before = len(df)
df.drop_duplicates(subset=[TEXT_COL], keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Rows removed (duplicates): {before - len(df):,}  |  Remaining: {len(df):,}')

### 3.4 Text Length Statistics

In [ ]:
df['char_len']  = df[TEXT_COL].str.len()
df['word_len']  = df[TEXT_COL].str.split().str.len()

stats = df[['char_len', 'word_len']].describe(percentiles=[.25, .5, .75, .90, .95, .99])
print('Text length statistics')
print(stats.round(1).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Character length — overall
axes[0, 0].hist(df['char_len'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0, 0].axvline(df['char_len'].median(), color='tomato', lw=1.8, label=f'Median: {df["char_len"].median():.0f}')
axes[0, 0].set_title('Character Length Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Characters'); axes[0, 0].set_ylabel('Count'); axes[0, 0].legend()

# Word length — overall
axes[0, 1].hist(df['word_len'], bins=40, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[0, 1].axvline(df['word_len'].median(), color='tomato', lw=1.8, label=f'Median: {df["word_len"].median():.0f}')
axes[0, 1].set_title('Word Length Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Words'); axes[0, 1].set_ylabel('Count'); axes[0, 1].legend()

# Character length per class
for cls, grp in df.groupby(LABEL_COL):
    axes[1, 0].hist(grp['char_len'], bins=50, alpha=0.55, label=str(cls), edgecolor='white')
axes[1, 0].set_title('Character Length by Class', fontweight='bold')
axes[1, 0].set_xlabel('Characters'); axes[1, 0].set_ylabel('Count'); axes[1, 0].legend()

# Word length per class — box plot
classes = df[LABEL_COL].unique()
data_per_class = [df.loc[df[LABEL_COL] == cls, 'word_len'].values for cls in sorted(classes)]
bp = axes[1, 1].boxplot(data_per_class, labels=[str(c) for c in sorted(classes)],
                         patch_artist=True, notch=True)
colors = sns.color_palette('muted', len(classes))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1, 1].set_title('Word Length by Class (Box Plot)', fontweight='bold')
axes[1, 1].set_xlabel('Class'); axes[1, 1].set_ylabel('Words')

plt.suptitle('Text Length Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'text_length_analysis.png', bbox_inches='tight')
plt.show()
print('Figure saved → output/text_length_analysis.png')

In [ ]:
# BERT coverage analysis — what % of texts fit within MAX_LENGTH tokens?
# (Quick proxy using whitespace-split word count)
coverage = (df['word_len'] <= MAX_LENGTH).mean() * 100
pct_truncated = 100 - coverage
print(f'Estimated coverage at MAX_LENGTH={MAX_LENGTH}: {coverage:.1f}% of samples')
print(f'Estimated truncated texts               : {pct_truncated:.1f}%')
if pct_truncated > 5:
    p95 = int(df['word_len'].quantile(0.95))
    print(f'💡  Consider MAX_LENGTH ≥ {p95} to cover 95% of samples.')

## 4. Text Cleaning

In [ ]:
# ─── Compiled patterns (define once for speed) ────────────────────────────────
_RE_URL     = re.compile(r'https?://\S+|www\.\S+')
_RE_MENTION = re.compile(r'@\w+')
_RE_HASHTAG = re.compile(r'#(\w+)')          # keep word, strip #
_RE_HTML    = re.compile(r'<[^>]+>')
_RE_EMOJI   = re.compile(
    '['
    u'\U0001F600-\U0001F64F'
    u'\U0001F300-\U0001F5FF'
    u'\U0001F680-\U0001F6FF'
    u'\U0001F1E0-\U0001F1FF'
    u'\U00002702-\U000027B0'
    u'\U000024C2-\U0001F251'
    ']+', flags=re.UNICODE)
_RE_REPEAT  = re.compile(r'(.)\1{2,}')       # 'looooove' → 'loove'
_RE_SPACE   = re.compile(r'\s+')


def clean_text(text: str) -> str:
    """Apply a reproducible cleaning pipeline to a single string."""
    text = str(text)
    text = _RE_HTML.sub(' ', text)            # strip HTML tags
    text = _RE_URL.sub('[URL]', text)         # replace URLs
    text = _RE_MENTION.sub('[USER]', text)    # replace @mentions
    text = _RE_HASHTAG.sub(r'\1', text)       # '#word' → 'word'
    text = _RE_EMOJI.sub(' ', text)           # remove emoji
    text = _RE_REPEAT.sub(r'\1\1', text)      # reduce char repetitions
    text = text.lower()                       # lowercase
    text = _RE_SPACE.sub(' ', text).strip()   # normalise whitespace
    return text


# Smoke test
sample = 'Loooove @elonmusk!! Check https://example.com #AI is 🚀🔥 <b>bold</b>'
print('Before:', sample)
print('After :', clean_text(sample))

In [ ]:
print('Cleaning text …', end=' ')
df['clean_text'] = df[TEXT_COL].apply(clean_text)

# Drop rows that became empty after cleaning
before = len(df)
df = df[df['clean_text'].str.strip() != ''].copy()
df.reset_index(drop=True, inplace=True)
print('done.')
print(f'Rows removed (empty after cleaning): {before - len(df):,}  |  Remaining: {len(df):,}')

In [ ]:
# Before / after length comparison
df['clean_char_len'] = df['clean_text'].str.len()
df['clean_word_len'] = df['clean_text'].str.split().str.len()

comparison = pd.DataFrame({
    'Raw chars'  : df['char_len'].describe().round(1),
    'Clean chars': df['clean_char_len'].describe().round(1),
    'Raw words'  : df['word_len'].describe().round(1),
    'Clean words': df['clean_word_len'].describe().round(1),
})
print('Before vs. After Cleaning')
print(comparison.to_string())

# Side-by-side sample
print('\n── Sample rows ──')
for _, row in df.sample(3, random_state=1).iterrows():
    print(f'  RAW  : {str(row[TEXT_COL])[:120]}')
    print(f'  CLEAN: {row["clean_text"][:120]}')
    print()

## 5. BERT Tokenization

In [ ]:
print(f'Loading tokenizer: {BERT_MODEL} …', end=' ')
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
print('done.')

# Vocabulary size
print(f'Vocab size     : {tokenizer.vocab_size:,}')
print(f'Padding token  : {tokenizer.pad_token!r}  (id={tokenizer.pad_token_id})')
print(f'CLS/SEP tokens : {tokenizer.cls_token!r} / {tokenizer.sep_token!r}')

In [ ]:
# ── Token-length analysis (sample for speed) ──────────────────────────────────
ANALYSIS_SAMPLE = min(5_000, len(df))
sample_df = df.sample(ANALYSIS_SAMPLE, random_state=RANDOM_STATE)

print(f'Counting token lengths on {ANALYSIS_SAMPLE:,} samples …', end=' ')
token_lengths = [
    len(tokenizer.encode(t, add_special_tokens=True))
    for t in sample_df['clean_text']
]
print('done.')

tl_series = pd.Series(token_lengths)
print(tl_series.describe(percentiles=[.5, .75, .90, .95, .99]).round(1).to_string())
print(f'\nFraction within MAX_LENGTH={MAX_LENGTH}: {(tl_series <= MAX_LENGTH).mean()*100:.1f}%')
print(f'Fraction truncated                     : {(tl_series > MAX_LENGTH).mean()*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Token length histogram
axes[0].hist(tl_series, bins=50, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[0].axvline(MAX_LENGTH, color='tomato', lw=2, linestyle='--',
                label=f'MAX_LENGTH={MAX_LENGTH}')
axes[0].axvline(tl_series.median(), color='gold', lw=1.5,
                label=f'Median={tl_series.median():.0f}')
axes[0].set_title('BERT Token Length Distribution', fontweight='bold')
axes[0].set_xlabel('Token count (incl. [CLS]/[SEP])')
axes[0].set_ylabel('Count'); axes[0].legend()

# CDF
sorted_tl = np.sort(tl_series)
cdf = np.arange(1, len(sorted_tl)+1) / len(sorted_tl)
axes[1].plot(sorted_tl, cdf * 100, color='mediumpurple', lw=2)
axes[1].axvline(MAX_LENGTH, color='tomato', lw=2, linestyle='--',
                label=f'MAX_LENGTH={MAX_LENGTH}')
axes[1].set_title('CDF of Token Lengths', fontweight='bold')
axes[1].set_xlabel('Token count'); axes[1].set_ylabel('Cumulative %')
axes[1].legend(); axes[1].set_ylim(0, 102)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'token_length_distribution.png', bbox_inches='tight')
plt.show()
print('Figure saved → output/token_length_distribution.png')

In [ ]:
# ── Encode full dataset ────────────────────────────────────────────────────────
BATCH_SIZE = 512
all_input_ids      = []
all_attention_mask = []
all_token_type_ids = []

texts = df['clean_text'].tolist()
total = len(texts)

print(f'Tokenizing {total:,} texts in batches of {BATCH_SIZE} …')
for i in range(0, total, BATCH_SIZE):
    batch = texts[i: i + BATCH_SIZE]
    enc = tokenizer(
        batch,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )
    all_input_ids.extend(enc['input_ids'])
    all_attention_mask.extend(enc['attention_mask'])
    all_token_type_ids.extend(enc['token_type_ids'])
    if (i // BATCH_SIZE) % 10 == 0:
        print(f'  {min(i+BATCH_SIZE, total):>8,} / {total:,}', end='\r')

df['input_ids']      = all_input_ids
df['attention_mask'] = all_attention_mask
df['token_type_ids'] = all_token_type_ids

print(f'\n✅ Tokenization complete — shape: {df.shape}')

## 6. Stratified Split  (70 / 15 / 15)

In [ ]:
def stratified_split(df, label_col, train_r, val_r, test_r, seed):
    """Stratified train / val / test split with exact ratio enforcement."""
    assert abs(train_r + val_r + test_r - 1.0) < 1e-9

    df_train, df_temp = train_test_split(
        df, test_size=(val_r + test_r), stratify=df[label_col], random_state=seed
    )
    val_frac_of_temp = val_r / (val_r + test_r)
    df_val, df_test = train_test_split(
        df_temp, test_size=(1 - val_frac_of_temp), stratify=df_temp[label_col], random_state=seed
    )
    return df_train, df_val, df_test


df_train, df_val, df_test = stratified_split(
    df, LABEL_COL, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, RANDOM_STATE
)

for name, part in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    n = len(part)
    pct = n / len(df) * 100
    print(f'{name:5} : {n:>8,}  ({pct:.1f}%)')

In [ ]:
# Verify stratification
print('Class distribution per split')
print('─' * 60)
ref = df[LABEL_COL].value_counts(normalize=True).sort_index()

rows = []
for cls in sorted(df[LABEL_COL].unique()):
    row = {'Class': cls, 'Full %': f'{ref[cls]*100:.2f}%'}
    for name, part in [('Train %', df_train), ('Val %', df_val), ('Test %', df_test)]:
        pct = (part[LABEL_COL] == cls).mean() * 100
        row[name] = f'{pct:.2f}%'
    rows.append(row)

pd.set_option('display.max_columns', None)
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
palette = sns.color_palette('muted', df[LABEL_COL].nunique())

for ax, (name, part) in zip(axes, [('Train', df_train), ('Validation', df_val), ('Test', df_test)]):
    counts = part[LABEL_COL].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color=palette, edgecolor='white')
    ax.set_title(f'{name}  (n={len(part):,})', fontweight='bold')
    ax.set_xlabel('Class'); ax.set_ylabel('Count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + len(part)*0.005,
                f'{bar.get_height():,.0f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Stratified Split — Class Distribution per Partition',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'split_distribution.png', bbox_inches='tight')
plt.show()
print('Figure saved → output/split_distribution.png')

## 7. Export Processed Dataset

In [ ]:
# ── Build label encoder ───────────────────────────────────────────────────────
unique_labels  = sorted(df[LABEL_COL].unique())
label2id       = {lbl: idx for idx, lbl in enumerate(unique_labels)}
id2label       = {idx: lbl for lbl, idx in label2id.items()}

print('Label encoding:', label2id)


def df_to_hf_dataset(part: pd.DataFrame) -> Dataset:
    """Convert a DataFrame partition to a HuggingFace Dataset."""
    return Dataset.from_dict({
        'input_ids':      part['input_ids'].tolist(),
        'attention_mask': part['attention_mask'].tolist(),
        'token_type_ids': part['token_type_ids'].tolist(),
        'labels':         [label2id[l] for l in part[LABEL_COL]],
        'clean_text':     part['clean_text'].tolist(),
        'raw_text':       part[TEXT_COL].tolist(),
    })


dataset_dict = DatasetDict({
    'train':      df_to_hf_dataset(df_train),
    'validation': df_to_hf_dataset(df_val),
    'test':       df_to_hf_dataset(df_test),
})

print(dataset_dict)

In [ ]:
# ── Save HuggingFace DatasetDict (Arrow format) ───────────────────────────────
HF_SAVE_PATH = OUTPUT_DIR / 'hf_dataset'
dataset_dict.save_to_disk(str(HF_SAVE_PATH))
print(f'✅ HuggingFace DatasetDict saved → {HF_SAVE_PATH}')

# ── Save as CSV (for inspection / non-HF workflows) ──────────────────────────
EXPORT_COLS = [TEXT_COL, 'clean_text', LABEL_COL,
               'char_len', 'word_len', 'clean_word_len']

for name, part in [('train', df_train), ('validation', df_val), ('test', df_test)]:
    out_path = OUTPUT_DIR / f'{name}.csv'
    part[EXPORT_COLS].to_csv(out_path, index=False)
    print(f'  {out_path}  ({len(part):,} rows)')

# ── Save label map ────────────────────────────────────────────────────────────
label_map_path = OUTPUT_DIR / 'label_map.json'
with open(label_map_path, 'w') as fh:
    json.dump({'label2id': label2id, 'id2label': id2label}, fh, indent=2)
print(f'  {label_map_path}')

In [ ]:
# ── Pipeline summary ──────────────────────────────────────────────────────────
print('=' * 58)
print('          NLP PREPROCESSING PIPELINE — SUMMARY')
print('=' * 58)
print(f'  Source dataset       : {KAGGLE_DATASET}')
print(f'  Total rows (clean)   : {len(df):,}')
print(f'  Text column          : {TEXT_COL}')
print(f'  Label column         : {LABEL_COL}')
print(f'  Classes              : {unique_labels}')
print(f'  BERT tokenizer       : {BERT_MODEL}')
print(f'  Max token length     : {MAX_LENGTH}')
print(f'─────────────────────────────────────────────────────')
print(f'  Train split          : {len(df_train):,}  ({len(df_train)/len(df)*100:.1f}%)')
print(f'  Validation split     : {len(df_val):,}  ({len(df_val)/len(df)*100:.1f}%)')
print(f'  Test split           : {len(df_test):,}  ({len(df_test)/len(df)*100:.1f}%)')
print(f'─────────────────────────────────────────────────────')
print(f'    hf_dataset/        (HuggingFace DatasetDict)')
print(f'    train.csv / validation.csv / test.csv')
print(f'    label_map.json')
print(f'    *.png              (EDA figures)')
print('=' * 58)

## 8. Reload & Verify (Sanity Check)

In [ ]:
from datasets import load_from_disk

reloaded = load_from_disk(str(HF_SAVE_PATH))
print('Reloaded DatasetDict:')
print(reloaded)

sample = reloaded['train'][0]
print('\nSample fields     :', list(sample.keys()))
print('input_ids[:10]    :', sample['input_ids'][:10])
print('attention_mask[:10]:', sample['attention_mask'][:10])
print('label             :', sample['labels'], '→', id2label[sample['labels']])
print('clean_text        :', sample['clean_text'][:80])